# Lab 07: Deploy Models & Monitor for Drift

> **Source:** Microsoft Learning -- [mslearn-mlops](https://github.com/MicrosoftLearning/mslearn-mlops), `docs/07-deploy-monitor.md`, `src/deploy_to_online_endpoint.py`

## Objectives

By the end of this lab you will be able to:

1. **Create** managed online endpoints for real-time inference
2. **Deploy** models with configurable instance types and counts
3. **Configure** traffic routing for blue-green deployments
4. **Set up** data collection on endpoints for monitoring
5. **Monitor** for data drift using Azure ML Model Monitor

| Estimated Time | Estimated Cost |
|---|---|
| ~45 minutes | ~$2-5 |

## Architecture Overview

![Architecture](architecture.png)

Managed online endpoints provide a production-grade serving layer for ML models. The endpoint abstracts away infrastructure concerns and exposes a REST API. Blue-green deployments allow safe rollouts -- deploy a new model version as "green" while "blue" continues serving traffic, then gradually shift traffic once validated. Model monitoring continuously compares production data distributions against training baselines to detect drift.

## Prerequisites

Run the cell below to verify the Azure ML SDK is installed and authenticate to your workspace.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)
print(f"Connected to workspace: {ml_client.workspace_name}")

## Step 1: Create a Managed Online Endpoint

A **Managed Online Endpoint** is Azure ML's abstraction for real-time inference. It handles:
- Load balancing across deployment instances
- Authentication (key-based or Azure AD token)
- Autoscaling and health monitoring
- Blue-green traffic routing

Key parameters:
- `auth_mode="key"` -- clients authenticate with a primary/secondary key (simpler)
- `auth_mode="aml_token"` -- clients use Azure AD tokens (more secure, recommended for production)

> **Exam Tip:** Managed online endpoints bill per-hour for each provisioned instance, even with zero traffic. Always delete test endpoints when done to avoid unexpected charges.

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import datetime

endpoint_name = f"diabetes-ep-{datetime.datetime.now().strftime('%m%d%H%M')}"
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Diabetes classification endpoint",
    auth_mode="key",
)
ml_client.begin_create_or_update(endpoint).result()
print(f"Endpoint created: {endpoint_name}")

## Step 2: Deploy a Model

A **ManagedOnlineDeployment** hosts a specific model version behind the endpoint. Key configuration:

- `model` -- the model artifact (MLflow format recommended for automatic inference)
- `instance_type` -- VM size (e.g., `Standard_D2as_v4` for CPU inference)
- `instance_count` -- number of instances for load distribution

MLflow models provide automatic scoring scripts, so you don't need to write custom inference code.

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment, Model
from azure.ai.ml.constants import AssetTypes

model = Model(path="./model", type=AssetTypes.MLFLOW_MODEL, description="Diabetes classification model")
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=model,
    instance_type="Standard_D2as_v4",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Blue deployment created")

## Step 3: Route Traffic

Traffic routing controls how requests are distributed across deployments. The `traffic` dictionary maps deployment names to integer percentages that must sum to 100.

Common patterns:
- `{"blue": 100}` -- all traffic to the current production deployment
- `{"blue": 90, "green": 10}` -- canary release: 10% of traffic tests the new model
- `{"green": 100}` -- full cutover to the new model

> **Exam Tip:** The `traffic` dictionary maps deployment names to percentages. This is how Azure ML implements blue-green and canary deployment strategies for managed online endpoints.

In [ ]:
endpoint = ml_client.online_endpoints.get(endpoint_name)
endpoint.traffic = {"blue": 100}
ml_client.begin_create_or_update(endpoint).result()
print(f"Traffic set to blue: 100%")
print(f"Scoring URI: {endpoint.scoring_uri}")

## Step 4: Test the Endpoint

Once the endpoint is deployed and traffic is routed, you can send inference requests. The `invoke` method sends data to the endpoint and returns the model's prediction.

In [ ]:
import json

sample_data = {
    "input_data": {
        "columns": ["Pregnancies", "PlasmaGlucose", "DiastolicBloodPressure",
                     "TricepsThickness", "SerumInsulin", "BMI",
                     "DiabetesPedigree", "Age"],
        "data": [[1, 85, 66, 29, 0, 26.6, 0.351, 31]]
    }
}
response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    request_file=None,
    deployment_name="blue",
)
print(f"Prediction: {response}")

## Step 5: Model Monitoring & Drift Detection

Once a model is deployed, its performance can degrade over time as the real-world data distribution shifts away from what the model was trained on. This is called **data drift**.

### Data Collection

Azure ML can automatically collect inference data from managed endpoints using the `DataCollector` configuration. This logs:
- Input features sent to the model
- Model predictions
- Timestamps for each request

### Drift Detection

Azure ML Model Monitor compares the distribution of production data against a **baseline** (typically the training dataset). It uses statistical measures such as:

- **Normalized Wasserstein Distance** -- measures the "cost" of transforming one distribution into another
- **Population Stability Index (PSI)** -- quantifies how much a distribution has shifted
- **Jensen-Shannon Divergence** -- symmetric measure of distribution similarity

When drift exceeds a configurable threshold, alerts are triggered (e.g., email, webhook, or Azure Monitor).

### Setting up monitoring (CLI)

```bash
az ml schedule create --file monitor-config.yml \
  --resource-group <rg> --workspace-name <ws>
```

> **Exam Tip:** Model monitoring compares production data distribution against the training data baseline. Know the key metrics (Wasserstein distance, PSI) and that monitoring runs on a schedule (not real-time). Threshold-based alerts notify teams when drift is significant.

## Step 6: Blue-Green Deployment Pattern

Blue-green deployment is the standard pattern for safe model updates in production:

### The process

1. **Blue** (current) is serving 100% of traffic
2. Deploy the new model version as **Green** alongside Blue
3. Route a small percentage of traffic to Green (canary: `{"blue": 90, "green": 10}`)
4. Monitor Green's performance (latency, errors, prediction quality)
5. If Green looks good, shift all traffic: `{"green": 100}`
6. Delete the old Blue deployment to stop billing

### Rollback

If Green shows problems at any point:
```python
endpoint.traffic = {"blue": 100, "green": 0}
ml_client.begin_create_or_update(endpoint).result()
```
Traffic instantly returns to the stable deployment with zero downtime.

> **Exam Tip:** Blue-green deployments provide zero-downtime rollbacks. The traffic dictionary is the mechanism for gradual migration. Both deployments run simultaneously during the transition period, so you pay for both until you delete the old one.

## Cleanup

**IMPORTANT:** Managed online endpoints bill per-hour for each provisioned instance, even with zero traffic. Always delete test endpoints when you are done.

```python
ml_client.online_endpoints.begin_delete(name=endpoint_name)
```

Run the cell below to delete the endpoint and stop billing.

In [ ]:
ml_client.online_endpoints.begin_delete(name=endpoint_name)
print(f"Endpoint '{endpoint_name}' deletion initiated. This may take a few minutes.")

## Key Takeaways

1. **Managed online endpoints** -- production-grade REST API serving layer with built-in load balancing, authentication, and autoscaling
2. **Blue-green deployments** -- deploy new model versions alongside the current one, then gradually shift traffic for safe rollouts
3. **Traffic routing** -- the `traffic` dictionary controls request distribution across deployments (percentages must sum to 100)
4. **Data collection** -- automatically log inference inputs and predictions for monitoring and auditing
5. **Drift monitoring** -- compare production data distributions against training baselines using statistical measures (Wasserstein distance, PSI)
6. **DELETE endpoints after testing** -- managed endpoints bill continuously for provisioned instances; always clean up

> **Summary:** This lab covered the full deployment lifecycle -- from creating endpoints and deployments, to routing traffic, testing inference, monitoring for drift, and cleaning up resources.